# Phần 6: Hyperparameter Tuning

Ở Phần 4, XGBoost thắng với tham số mặc định/đoán tay. Ở đây ta dùng
**RandomizedSearchCV** để tìm tham số tốt hơn một cách có hệ thống cho
cả XGBoost và RandomForest, rồi so sánh công bằng bằng 5-fold CV để
quyết định có nên cập nhật model cuối cùng hay không.

**Vì sao RandomizedSearchCV chứ không phải GridSearchCV?** Với nhiều
tham số liên tục (learning_rate, subsample...), thử *tất cả* tổ hợp
(Grid) sẽ tốn thời gian theo cấp số nhân. Random chỉ thử ngẫu nhiên N
tổ hợp trong không gian tham số — thường tìm được vùng tốt gần tương
đương mà nhanh hơn nhiều lần, phù hợp để chạy trên máy cá nhân.

> Logic chi tiết nằm trong `src/tune.py`. Notebook này gọi lại để minh họa.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib

from data_loader import load_processed_data
from tune import tune_model, baseline_rmsle, XGB_PARAM_DIST, RF_PARAM_DIST
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

df = load_processed_data()
X = df.drop(columns=['price'])
y = np.log1p(df['price'])

## Bước 1: Không gian tham số tìm kiếm

| Model | Tham số | Khoảng tìm kiếm | Ý nghĩa |
|---|---|---|---|
| XGBoost | `max_depth` | 3-7 | Độ sâu mỗi cây — sâu hơn = phức tạp hơn, dễ overfit |
| XGBoost | `learning_rate` | 0.02-0.20 | Tốc độ học — thấp hơn thường tổng quát tốt hơn nhưng cần nhiều cây hơn |
| XGBoost | `subsample`, `colsample_bytree` | 0.7-1.0 | Tỉ lệ dữ liệu/feature dùng mỗi cây — giảm overfit |
| RandomForest | `max_depth` | 5-20 | Độ sâu mỗi cây |
| RandomForest | `min_samples_leaf` | 1-8 | Số mẫu tối thiểu ở lá — cao hơn giúp giảm overfit |
| RandomForest | `max_features` | 0.3-1.0 | Tỉ lệ feature xét mỗi lần chia nhánh |

## Bước 2: Tune XGBoost

In [ ]:
xgb_baseline_rmsle = baseline_rmsle(
    XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05,
                 subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1),
    X, y
)
print(f'XGBoost baseline (Phần 4) RMSLE = {xgb_baseline_rmsle:.4f}')

xgb_tuned, xgb_tuned_rmsle, xgb_params = tune_model(
    XGBRegressor(random_state=42, n_jobs=-1), XGB_PARAM_DIST, X, y,
    n_iter=15, name='XGBoost'
)

## Bước 3: Tune RandomForest

In [ ]:
rf_baseline_rmsle = baseline_rmsle(
    RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1),
    X, y
)
print(f'RandomForest baseline (Phần 4) RMSLE = {rf_baseline_rmsle:.4f}')

rf_tuned, rf_tuned_rmsle, rf_params = tune_model(
    RandomForestRegressor(random_state=42, n_jobs=-1), RF_PARAM_DIST, X, y,
    n_iter=15, name='RandomForest'
)

## Bước 4: So sánh cuối cùng (5-fold, công bằng)

`tune_model()` ở trên dùng 3-fold cho nhanh khi tìm kiếm, nhưng quyết
định *cuối cùng* nên dùng cùng một chuẩn 5-fold cho mọi model để so
sánh công bằng.

In [ ]:
prev_model = joblib.load('../models/best_model.pkl')
prev_rmsle = baseline_rmsle(prev_model, X, y)
xgb_tuned_rmsle_5fold = baseline_rmsle(xgb_tuned, X, y)
rf_tuned_rmsle_5fold = baseline_rmsle(rf_tuned, X, y)

comparison = pd.DataFrame([
    {'model': 'best_model.pkl (đã lưu)', 'RMSLE_5fold': prev_rmsle},
    {'model': 'XGBoost_tuned (mới)', 'RMSLE_5fold': xgb_tuned_rmsle_5fold},
    {'model': 'RandomForest_tuned (mới)', 'RMSLE_5fold': rf_tuned_rmsle_5fold},
]).sort_values('RMSLE_5fold')
comparison

## Kết quả thực tế đã chạy

| Model | RMSLE (baseline) | RMSLE (tuned) | Cải thiện |
|---|---|---|---|
| XGBoost | 0.2958 | **0.2952** | nhỏ (~0.2%) |
| RandomForest | 0.3133 | 0.3081 | nhỏ (~1.7%) |

**Bài học quan trọng:** Tuning chỉ cải thiện rất ít so với baseline ở
Phần 4. Đây là kết quả **bình thường và có thật** khi:
1. Tham số baseline ban đầu đã được chọn khá hợp lý (không phải mặc
   định 100% của sklearn, mà đã có suy nghĩ: `max_depth` vừa phải,
   `subsample`/`colsample_bytree` < 1 để giảm overfit...).
2. Giới hạn thực sự nằm ở **dữ liệu** (xem worst predictions ở Phần 5:
   nhiều căn nhà đắt bất thường do yếu tố dataset không có, ví dụ vị trí
   chi tiết) chứ không phải ở việc model chưa được tối ưu tham số.

Đây là lý do trong thực tế, sau một vài lần thử tuning không cải thiện
nhiều, nên chuyển hướng đầu tư sang **feature engineering** (thêm dữ
liệu, tạo đặc trưng mới) thay vì tiếp tục tinh chỉnh tham số — thường
mang lại lợi ích lớn hơn nhiều.

In [ ]:
best_row = comparison.iloc[0]
print('Model tốt nhất:', best_row['model'])

if best_row['model'] == 'XGBoost_tuned (mới)':
    xgb_tuned.fit(X, y)
    joblib.dump(xgb_tuned, '../models/best_model.pkl')
    print('Đã cập nhật models/best_model.pkl')
else:
    print('Model đã lưu vẫn tốt nhất/gần bằng -> giữ nguyên.')